In [1]:
from pathlib import Path
from collections import defaultdict
import random
import sounddevice as sd
import parselmouth
from pathlib import Path

from sample_generation import LIBRISPEECH_DIR

root = LIBRISPEECH_DIR / "dev-clean"
CLIPS = defaultdict(lambda: defaultdict(list))  # speaker -> chapter -> files

for f in root.glob("*/*/*.flac"):
    speaker, chapter = f.parent.parent.name, f.parent.name
    CLIPS[int(speaker)][int(chapter)].append(f)

def play_audio(speaker_id, chapter_id, index, clips = CLIPS):
    snd = parselmouth.Sound(str(clips[speaker_id][chapter_id][index]))
    sd.play(snd.values.T, snd.sampling_frequency)
    sd.wait()

#play_audio(2412, 153954, 13)

objc[59387]: Class AVFFrameReceiver is implemented in both /Users/jscheltema/Library/Caches/pypoetry/virtualenvs/pal-5szAi3Ow-py3.12/lib/python3.12/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x105f943a8) and /Users/jscheltema/Library/Caches/pypoetry/virtualenvs/pal-5szAi3Ow-py3.12/lib/python3.12/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x1247743a8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[59387]: Class AVFAudioReceiver is implemented in both /Users/jscheltema/Library/Caches/pypoetry/virtualenvs/pal-5szAi3Ow-py3.12/lib/python3.12/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x105f943f8) and /Users/jscheltema/Library/Caches/pypoetry/virtualenvs/pal-5szAi3Ow-py3.12/lib/python3.12/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x1247743f8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
/Users/jscheltema/Li

In [2]:
from utils import transform_data
from data_generation import create_train_data
from sample_generation import load_samples, SAMPLES_DIR

data = load_samples(SAMPLES_DIR / "librispeech_25000.json")
test_data = load_samples(SAMPLES_DIR / "synthetic_2500.json")

train_dl, val_dl, mean, std = create_train_data(data)
X_test, y_test = transform_data(test_data, mean, std)

In [7]:
from encoder import SpeakerEncoder, SmallSpeakerEncoder
from classification import train_encoder, test_encoder, train_xgb
from sample_generation import librispeech_files_by_speaker

xgb = train_xgb(data, mean, std)

encoder = train_encoder(librispeech_files_by_speaker("dev-clean"), SmallSpeakerEncoder, epochs=10)

val acc  0.882
epoch   0  loss 2.9038  val speaker acc 0.288
epoch   1  loss 2.4726  val speaker acc 0.428
epoch   2  loss 1.7849  val speaker acc 0.592
epoch   3  loss 1.2925  val speaker acc 0.748
epoch   4  loss 0.9320  val speaker acc 0.804
epoch   5  loss 0.6569  val speaker acc 0.808
epoch   6  loss 0.7619  val speaker acc 0.840
epoch   7  loss 0.5208  val speaker acc 0.840
epoch   8  loss 0.3979  val speaker acc 0.844
epoch   9  loss 0.3443  val speaker acc 0.912


In [8]:
# Synthetic voices are speakers the encoder never saw during training
sims = test_encoder(encoder, test_data, threshold=0.795)

threshold 0.795
accuracy  0.824
precision 0.765
recall    0.939
confusion: TP=1183 FP=364 TN=876 FN=77
best threshold 0.864 -> accuracy 0.851


In [11]:
from encoder import load_wave, log_mel

job_path = '/Users/jscheltema/Documents/Personal/PAL/pal/perception/audio/voice_analysis/data/mine/job_voice.wav'
mar_path = '/Users/jscheltema/Documents/Personal/PAL/pal/perception/audio/voice_analysis/data/mine/marijn_voice.wav'

job_mel = log_mel(load_wave(job_path)).unsqueeze(0)  # (1, 80, frames)
job_embedding = encoder.embedding(job_mel)

mar_mel = log_mel(load_wave(mar_path)).unsqueeze(0)  # (1, 80, frames)
mar_embedding = encoder.embedding(mar_mel)

In [9]:
import time
import numpy as np
import pandas as pd
import parselmouth
import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

from data_generation import extract_voice_features
from encoder import load_wave, log_mel

ENCODER_THRESHOLD = 0.864  # same threshold as the test_encoder cell
N_CLIPS = 50               # clips to time per model
encoder.eval()

# _________ Accuracy: same synthetic test pairs for both (speakers neither model trained on) _________

y_true = y_test.numpy().astype(bool)
xgb_score = xgb.predict_proba(X_test.numpy())[:, 1]
enc_score = sims.numpy()   # cosine similarities from the test_encoder cell

def quality(score, threshold):
    pred = score > threshold
    return {
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred),
        "recall": recall_score(y_true, pred),
        "auc": roc_auc_score(y_true, score),  # threshold independent
    }

# _________ Latency _________

def median_ms(fn, *args, repeats=3):
    fn(*args)  # warm-up
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        fn(*args)
        times.append((time.perf_counter() - start) * 1000)
    return float(np.median(times))

features_uncached = extract_voice_features.__wrapped__  # bypass lru_cache, otherwise repeats cost ~0 ms

def embed(path):
    with torch.inference_mode():
        return encoder.embedding(log_mel(load_wave(path)).unsqueeze(0))[0]

clips = list(dict.fromkeys(p for s in test_data for p in (s.sample_1, s.sample_2)))[:N_CLIPS]
durations = np.array([parselmouth.Sound(str(c)).get_total_duration() for c in clips])

xgb_clip_ms = np.array([median_ms(features_uncached, c) for c in clips])
enc_clip_ms = np.array([median_ms(embed, c) for c in clips])

# Decision step once both clips are processed: pair features + tree model vs. one cosine similarity
pair = test_data[0]
e1, e2 = embed(pair.sample_1), embed(pair.sample_2)
xgb_decide_ms = median_ms(lambda: xgb.predict_proba(transform_data([pair], mean, std)[0].numpy()), repeats=100)
enc_decide_ms = median_ms(lambda: F.cosine_similarity(e1, e2, dim=0) > ENCODER_THRESHOLD, repeats=100)

def latency(clip_ms, decide_ms):
    return {
        "ms/clip (median)": np.median(clip_ms),
        "ms/clip (p95)": np.percentile(clip_ms, 95),
        "ms per s audio": np.median(clip_ms / durations),
        "ms decision": decide_ms,
        # live use: the reference voice is processed once, so a new utterance costs one clip + the decision
        "ms per new utterance": np.median(clip_ms) + decide_ms,
    }

results = pd.DataFrame({
    "xgb (praat features)": {**quality(xgb_score, 0.5), **latency(xgb_clip_ms, xgb_decide_ms)},
    "encoder (embeddings)": {**quality(enc_score, ENCODER_THRESHOLD), **latency(enc_clip_ms, enc_decide_ms)},
}).T

print(f"{len(y_true)} test pairs, {len(clips)} timed clips ({durations.mean():.1f} s average)")
results.round(3)


2500 test pairs, 50 timed clips (3.0 s average)


,accuracy,precision,recall,auc,ms/clip (median),ms/clip (p95),ms per s audio,ms decision,ms per new utterance
xgb (praat features),0.856,0.806,0.940,0.943,33.145,56.385,12.160,12.636,45.781
encoder (embeddings),0.850,0.837,0.873,0.925,9.196,14.468,2.896,0.007,9.203


In [ ]:
from encoder import save_encoder, load_encoder, enroll

# Save after training (stores weights, model sizes, threshold and audio settings)
save_encoder(encoder, threshold=0.864)

NameError: name 'encoder' is not defined

In [3]:
from encoder import save_encoder, load_encoder, enroll


# Load again, e.g. in a fresh kernel or another script
encoder, threshold = load_encoder()

In [10]:
from encoder import enroll
import torch

me = enroll(["/Users/jscheltema/Documents/Personal/PAL/pal/perception/audio/voice_analysis/data/mine/marijn_voice.wav"], encoder)
torch.save(me, "voiceprint_marijn.pt")  # store once, reuse for every check

In [9]:
from encoder import verify
import torch

voiceprint = '/Users/jscheltema/Documents/Personal/PAL/pal/perception/audio/voice_analysis/data/voiceprint/voiceprint_me.pt'

audio = '/Users/jscheltema/Documents/Personal/PAL/pal/perception/audio/voice_analysis/data/mine/marijn_voice.wav'

verify(torch.load(voiceprint), audio, encoder, 0.865)[0]


False